# 🌫️ 实验13：Diffusion Policy（扩散策略）on PushT

上一个实验我们用 **ACT** 做行为克隆——直接把观测**回归**成动作块。
这次换 **Diffusion Policy（DP）**，并用它的**经典基准任务 PushT**：用一个小圆推头，把桌面上的 **T 形块推到 T 形目标位置**。

DP 生成动作的方式和 ACT 完全不同——它**学“去噪”**：训练时给专家动作加噪声、让网络预测噪声；推理时从**纯噪声**出发，在观测条件下**迭代去噪**，一步步还原出一段动作。

| | ACT | Diffusion Policy |
|---|---|---|
| 生成方式 | 一次前向**直接回归**动作块 | 从噪声**迭代去噪**（这里 100 步 DDPM）|
| 观测 | 1 帧 | **2 帧历史** |
| 主干 | Transformer 编解码器 + CVAE | **1D 卷积 UNet**（去噪器）+ 视觉编码 |
| 损失 | $\ell_1$ 重构 $+$ KL | **预测噪声的 MSE** |

> 标注 `# TODO（填空 N）` 的地方需要你补全代码后才能运行。

## 0. 环境安装

和 ACT 实验**完全一样**的环境即可——`diffusers` 已随 `lerobot` 一起装好，扩散策略开箱即用。

```powershell
conda create -n lerobot python=3.12 -y
conda activate lerobot
pip install torch==2.10.0 torchvision==0.25.0 --index-url https://download.pytorch.org/whl/cu128
pip install "lerobot[dataset]" matplotlib
```

> PushT 图像很小（96×96），比 ACT 的 SO-101（双目 480×640）轻不少，**batch 8 也只占 ~5 GB 显存**，适配 8 GB 显卡。

In [ ]:
import torch
import lerobot
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as IPyImage

from lerobot.configs.types import FeatureType
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.utils.feature_utils import dataset_to_policy_features
from lerobot.utils.constants import ACTION
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy
from lerobot.policies.factory import make_pre_post_processors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("lerobot:", lerobot.__version__, "| torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("设备:", device)

---
## 第一部分：认识数据集（PushT）

先看看数据长什么样：一路 **96×96** 的俯视图、**2 维**的推头位置（state）、**2 维**的动作（要移动到的 x,y 目标）。

In [ ]:
dataset_id = "lerobot/pusht"
meta = LeRobotDatasetMetadata(dataset_id)
print("轨迹数:", meta.total_episodes, "| 总帧数:", meta.total_frames, "| fps:", meta.fps)
print("相机:", meta.camera_keys, "| 动作维度:", meta.features["action"]["shape"])

explore_ds = LeRobotDataset(dataset_id, video_backend="pyav")
sample = explore_ds[0]
print("任务:", sample["task"])
for k in ["observation.image", "observation.state", "action"]:
    print(f"  {k:20s}{tuple(sample[k].shape)}")

### 看画面 + 播放一条轨迹（动图）

In [ ]:
def to_img(t):
    a = (t.numpy().transpose(1, 2, 0) * 255).clip(0, 255).astype("uint8")
    return Image.fromarray(a)

print("单帧画面（96×96，放大显示）：灰色是推头，蓝色 T 是要推的块，绿色 T 是目标")
display(to_img(sample["observation.image"]).resize((240, 240), Image.NEAREST))

ep = 0
row = meta.episodes[ep]
start, end = int(row["dataset_from_index"]), int(row["dataset_to_index"])
frames = [to_img(explore_ds[gi]["observation.image"]).resize((240, 240), Image.NEAREST) for gi in range(start, end, 2)]
frames[0].save("pusht_episode.gif", save_all=True, append_images=frames[1:], duration=60, loop=0)
print(f"\n轨迹 {ep}：{end - start} 帧")
IPyImage("pusht_episode.gif")

### 看动作数据：推头的 2D 运动轨迹
PushT 的状态/动作都是 **2 维**。把这条轨迹里推头的位置画成 x–y 路径，就能看到「怎么推」。

In [ ]:
pos = np.stack([explore_ds[gi]["observation.state"].numpy() for gi in range(start, end)])
plt.figure(figsize=(4.5, 4.5))
plt.plot(pos[:, 0], pos[:, 1], "-o", ms=2, lw=1)
plt.scatter(*pos[0], c="g", s=80, label="起点", zorder=3)
plt.scatter(*pos[-1], c="r", s=80, label="终点", zorder=3)
plt.gca().invert_yaxis(); plt.gca().set_aspect("equal")
plt.title(f"轨迹 {ep}：推头 2D 运动路径"); plt.legend(); plt.tight_layout(); plt.show()

---
## 第二部分：构建 Diffusion Policy

用数据集 metadata 划分输入/输出特征，再建配置与策略。

**提示**：① 动作特征 `ft.type` 是 `FeatureType.ACTION`；② `DiffusionConfig(input_features=?, output_features=?)`；③ 动作用 `cfg.action_delta_indices`。
DP 的数据需要 **`observation.state` 的历史**（`observation_delta_indices = [-1, 0]`）。

In [ ]:
features = dataset_to_policy_features(meta.features)

# TODO（填空 1）：选出“动作”特征作为输出
output_features = {k: ft for k, ft in features.items() if ...}        # ← 修改 ...
input_features = {k: ft for k, ft in features.items() if k not in output_features}

# TODO（填空 2）：用上面的特征构建扩散策略配置
cfg = DiffusionConfig(input_features=..., output_features=...)        # ← 填入两个参数

policy = DiffusionPolicy(cfg).to(device)
preprocess, postprocess = make_pre_post_processors(cfg, dataset_stats=meta.stats)

def make_delta(idx, fps):
    return [0] if idx is None else [i / fps for i in idx]
delta = {
    "observation.state": make_delta(cfg.observation_delta_indices, meta.fps),
    # TODO（填空 3）：动作要返回未来 horizon 步（填第一个参数）
    "action": make_delta(..., meta.fps),                             # ← 填入 ...
}
delta |= {k: make_delta(cfg.observation_delta_indices, meta.fps) for k in cfg.image_features}

dataset = LeRobotDataset(dataset_id, delta_timestamps=delta, video_backend="pyav")
loader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0, drop_last=True)
print("数据集就绪。")

### 打印模型关键结构（无需填空）
DP 的核心是 `policy.diffusion`，里面两块：**`rgb_encoder`**（把图像编码成条件）和 **`unet`**（真正的去噪器）。看一下配置和参数分布。

In [ ]:
print("n_obs_steps          :", cfg.n_obs_steps, " 用几帧历史观测")
print("horizon              :", cfg.horizon, " 一次预测多长的动作序列")
print("n_action_steps       :", cfg.n_action_steps, " 实际执行多少步再重规划")
print("num_train_timesteps  :", cfg.num_train_timesteps, " 扩散/去噪步数（DDPM）")
print("vision_backbone      :", cfg.vision_backbone, "| down_dims(UNet):", cfg.down_dims)

def count(m):
    return sum(p.numel() for p in m.parameters())
total = count(policy.diffusion)
print(f"\n{'子模块':<16}{'参数量':>14}{'占比':>9}")
print("-" * 40)
for name, child in policy.diffusion.named_children():
    print(f"{name:<16}{count(child):>14,}{100*count(child)/total:>8.1f}%")
print("-" * 40)
print(f"{'合计':<16}{total:>14,}")

**去噪的直觉（DDPM）**：
- **训练**：取一段专家动作 $a_0$，随机选噪声等级 $t\in[1,100]$，加高斯噪声得 $a_t$；`unet` 在「图像+状态」条件下**预测所加的噪声** $\hat\epsilon$，损失为 $\|\hat\epsilon-\epsilon\|^2$。
- **推理**：从纯噪声出发，反复预测噪声并去掉一点，**100 步**后得到干净动作。

---
## 第三部分：训练（去噪损失）

`policy.forward(batch)` 返回**预测噪声的 MSE**。补全反向传播三步。
扩散损失每步随机选噪声等级，**比 ACT 更抖**，看整体趋势即可。

**提示**：`optimizer.zero_grad()` / `loss.backward()` / `optimizer.step()`。

In [ ]:
def batches(loader):
    while True:
        for b in loader:
            yield b
stream = batches(loader)
def to_device(b):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}

policy.train()
optimizer = cfg.get_optimizer_preset().build(policy.parameters())
training_steps = 200
loss_history = []

for step in range(1, training_steps + 1):
    batch = to_device(preprocess(next(stream)))
    loss, _ = policy.forward(batch)          # 预测噪声的 MSE

    # TODO（填空 4）：反向传播三步
    ...                                      # ← 清空梯度
    ...                                      # ← 反向传播
    ...                                      # ← 更新参数

    loss_history.append(loss.item())
    if step % 20 == 0:
        recent = sum(loss_history[-20:]) / 20
        print(f"训练步 {step:4d} | 损失 {loss.item():6.4f} | 近20步均值 {recent:6.4f}")
print("训练完成。")

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(loss_history, alpha=0.4, label="每步")
k = 20
if len(loss_history) >= k:
    ma = np.convolve(loss_history, np.ones(k) / k, mode="valid")
    plt.plot(range(k - 1, len(loss_history)), ma, label=f"{k} 步滑动平均")
plt.xlabel("训练步"); plt.ylabel("去噪 MSE 损失"); plt.legend()
plt.title("Diffusion Policy 训练（PushT）"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 第四部分：推理：从噪声去噪出动作块

DP 内部维护一个**观测队列**：`select_action` 每次喂**单帧**观测，凑够 `n_obs_steps=2` 帧后跑一次 100 步去噪、
生成 `horizon` 步动作、保留 `n_action_steps` 步执行。**第一次调用最慢**（跑完整去噪），之后从队列快速取。

**提示**：用 `policy.select_action(dict(obs1))`。

In [ ]:
import time
policy.reset(); policy.eval()

raw = to_device(preprocess(next(stream)))
# 构造单帧观测：去掉时间维（取最后一帧），batch 取 1
obs1 = {}
for k, v in raw.items():
    if k == ACTION or k.endswith("_is_pad"):
        continue
    if torch.is_tensor(v) and v.dim() > 1 and ("state" in k or "image" in k):
        obs1[k] = v[:1, -1]                  # (1, ...) 去掉历史维
print("单帧观测:", {k: tuple(v.shape) for k, v in obs1.items() if torch.is_tensor(v)})

t0 = time.perf_counter()
with torch.no_grad():
    for i in range(5):
        # TODO（填空 5）：喂单帧观测，得到一步动作（注意传 dict(obs1)）
        action = ...                          # ← 调用 policy 的推理方法
        dt = time.perf_counter() - t0
        print(f"  第 {i} 次 select_action: 动作 {tuple(action.shape)} | 队列剩 {len(policy._queues['action'])} | 累计 {dt:.2f}s")
        t0 = time.perf_counter()
print("\n第 0 次最慢（跑了 100 步去噪生成整段动作），后面是从队列里直接取。")

## 小结：ACT vs Diffusion Policy

- **同**：都是从专家示范做**行为克隆**，都输出一段动作块。
- **异**：
  - ACT 一次前向**直接回归**动作；DP 从噪声**迭代去噪**（100 步），推理更慢但表达力更强、天生处理多模态。
  - ACT 用 1 帧观测 + CVAE；DP 用 2 帧历史 + 扩散过程。
  - DP 的参数几乎都在**去噪 UNet** 上（~96%）。

### 思考题
1. 把 `num_inference_steps` 调小（如 10，DDIM），推理变快，动作质量会怎样？
2. DP 的损失为什么比 ACT 抖？（提示：每步随机采一个噪声等级 $t$）
3. 同样的观测，DP 多跑几次推理（不同初始噪声），动作会一样吗？这说明了什么？